# 01. Data Cleaning & Standardization - Kabaddi Matches EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adarshgaurav1951/Kabaddi-Matches-EDA/blob/main/notebooks/01_cleaning.ipynb)

## Context & Objectives
Kabaddi is a high-tempo, contact sport featuring rapid raid cycles and complex defensive formations. A standard professional match consists of two 20-minute halves. In this notebook, we ingest raw match, event, player, and team tables, audit data quality, resolve schema discrepancies, eliminate duplicate event logs, and standardize entity nomenclature.

### Cleaning Workflow
1. **Raw Ingestion & Audit**: Load source tables from `data/raw/` and inspect shapes, data types, and null value distributions.
2. **Match Table Standardization**: Remove unneeded index columns, parse date formats, resolve player of the match names, and standardize venue and series names.
3. **Team Table Standardization**: Flatten dot-notation column names, impute missing values for zero-count stats, and unify franchise names.
4. **Player Table & Role Imputation**: Clean column artifacts, merge official player positions from tournament records, and impute missing roles based on career raid vs. tackle volume.
5. **Event Play-by-Play Cleaning**: Deduplicate simultaneous/repeated event logs, standardize boolean flags (`do_or_die`, `super_raid`, `super_tackle`), and handle structural nulls.
6. **Integrity Validation & Export**: Assert domain constraints and export clean datasets to `data/cleaned/`.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Google Colab environment setup
if 'google.colab' in sys.modules:
    print('Running on Google Colab! Setting up repository files...')
    if not os.path.exists('data/raw') and not os.path.exists('../data/raw'):
        os.system('git clone https://github.com/adarshgaurav1951/Kabaddi-Matches-EDA.git _temp_repo')
        os.system('mkdir -p ../data/raw ../data/cleaned')
        os.system('cp -r "_temp_repo/Documents/Kabaddi Matches EDA/data/raw/"* ../data/raw/ 2>/dev/null || cp -r "_temp_repo/data/raw/"* ../data/raw/ 2>/dev/null')
        os.system('rm -rf _temp_repo')

# Detect data directories
if os.path.exists('../data/raw'):
    RAW_DIR = '../data/raw'
    CLEAN_DIR = '../data/cleaned'
elif os.path.exists('data/raw'):
    RAW_DIR = 'data/raw'
    CLEAN_DIR = 'data/cleaned'
else:
    RAW_DIR = 'data/raw'
    CLEAN_DIR = 'data/cleaned'

os.makedirs(CLEAN_DIR, exist_ok=True)
print(f'Environment initialized. RAW_DIR: {RAW_DIR}, CLEAN_DIR: {CLEAN_DIR}')


Environment initialized. RAW_DIR: ../data/raw, CLEAN_DIR: ../data/cleaned


## 1. Load and Audit Raw Datasets
We load the atomic CSV tables selected from the raw data repository.

In [2]:
# Ingest raw CSVs
df_matches_raw = pd.read_csv(os.path.join(RAW_DIR, 'matches.csv'))
df_teams_raw = pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))
df_players_raw = pd.read_csv(os.path.join(RAW_DIR, 'players.csv'))
df_events_raw = pd.read_csv(os.path.join(RAW_DIR, 'events.csv'), low_memory=False)
df_roles_raw = pd.read_csv(os.path.join(RAW_DIR, 'player_roles.csv'))

print(f"Matches Raw Shape: {df_matches_raw.shape}")
print(f"Teams Raw Shape:   {df_teams_raw.shape}")
print(f"Players Raw Shape: {df_players_raw.shape}")
print(f"Events Raw Shape:  {df_events_raw.shape}")
print(f"Roles Raw Shape:   {df_roles_raw.shape}")


Matches Raw Shape: (684, 17)
Teams Raw Shape:   (1368, 34)
Players Raw Shape: (16892, 31)
Events Raw Shape:  (66568, 41)
Roles Raw Shape:   (391, 3)


## 2. Clean Matches Dataset
We drop legacy index columns, parse match dates, map `toss_winner` team ID to franchise name, resolve `player_name_of_the_match` from the player registry, and standardize venue and series names.

In [3]:
df_matches = df_matches_raw.copy()

# Drop unneeded index column
if 'Unnamed: 0' in df_matches.columns:
    df_matches = df_matches.drop(columns=['Unnamed: 0'])

# Parse date column
df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')

# Build team ID to name mapping dictionary
team_name_corrections = {
    'Dabang Delhi': 'Dabang Delhi K.C.',
    'Fire birds': 'Fire Birds',
    'Services SCB': 'Services SSCB'
}
team_id_to_name = df_teams_raw.groupby('id')['name'].last().to_dict()
team_id_to_name = {k: team_name_corrections.get(v, v).strip() for k, v in team_id_to_name.items()}

# Player ID to name mapping
player_id_to_name = df_players_raw.groupby('player_id')['player_name'].last().to_dict()

# Standardize toss winner name
df_matches['toss_winner_team_id'] = df_matches['toss_winner'].astype('int32')
df_matches['toss_winner_name'] = df_matches['toss_winner_team_id'].map(team_id_to_name)

# Resolve player of the match name
df_matches['player_id_of_the_match'] = df_matches['player_id_of_the_match'].astype('Int32')
df_matches['player_name_of_the_match'] = df_matches['player_id_of_the_match'].map(player_id_to_name)

# Standardize franchise names in home_team_name
df_matches['home_team_name'] = df_matches['home_team_name'].replace(team_name_corrections).str.strip()
df_matches['result'] = df_matches['result'].fillna('Not Recorded').str.strip()

# Downcast integer types
df_matches['match_id'] = df_matches['match_id'].astype('int32')
df_matches['series_id'] = df_matches['series_id'].astype('int16')
df_matches['venue_id'] = df_matches['venue_id'].astype('int16')

# Check nulls and duplicates
print("Matches null summary:")
print(df_matches.isnull().sum()[df_matches.isnull().sum() > 0])
print(f"Matches duplicates in match_id: {df_matches['match_id'].duplicated().sum()}")
df_matches[['match_id', 'date', 'series_name', 'toss_winner_name', 'player_name_of_the_match', 'venue_name']].head(5)


Matches null summary:
date                         22
player_id_of_the_match      109
player_name_of_the_match    109
home_team_id                433
home_team_name              433
dtype: int64
Matches duplicates in match_id: 0


,match_id,date,series_name,toss_winner_name,player_name_of_the_match,venue_name
0,1,2014-07-26,"Pro Kabaddi League Season 1, 2014",U Mumba,Anup Kumar,"DOME@NSCI SVP Stadium, Mumbai"
1,10,2014-07-31,"Pro Kabaddi League Season 1, 2014",Dabang Delhi K.C.,Jang Kun Lee,"Netaji Subhash Chandra Bose Indoor Stadium, Ko..."
2,100,2015-08-09,"Pro Kabaddi League, 2015",Patna Pirates,Kashiling Adake,"Thyagaraj Sports Complex, Delhi"
3,101,2015-08-10,"Pro Kabaddi League, 2015",Jaipur Pink Panthers,NaN,"Thyagaraj Sports Complex, Delhi"
4,102,2015-08-11,"Pro Kabaddi League, 2015",Puneri Paltan,Rajesh Narwal,"Thyagaraj Sports Complex, Delhi"


## 3. Clean Teams Dataset
We drop scraping artifacts (`state_of_play.*`), rename nested dot-notation columns to clean snake_case names, impute zero-counts for super raids and super tackles, and unify franchise names.

In [4]:
df_teams = df_teams_raw.copy()

# Drop scraping artifacts and index columns
cols_to_drop = [c for c in df_teams.columns if c.startswith('state_of_play') or c in ['Unnamed: 0', 'squad']]
df_teams = df_teams.drop(columns=cols_to_drop)

# Rename dot notation columns to clean snake_case
col_rename = {
    'id': 'team_id',
    'name': 'team_name',
    'stats.all_outs': 'all_outs_inflicted',
    'stats.declare': 'declare_count',
    'stats.points.all_out': 'all_out_points',
    'stats.points.declare': 'declare_points',
    'stats.points.extras': 'extra_points',
    'stats.points.raid_points.raid_bonus': 'raid_bonus_points',
    'stats.points.raid_points.total': 'raid_points_total',
    'stats.points.raid_points.touch': 'raid_touch_points',
    'stats.points.tackle_points.capture': 'tackle_capture_points',
    'stats.points.tackle_points.capture_bonus': 'tackle_capture_bonus_points',
    'stats.points.tackle_points.total': 'tackle_points_total',
    'stats.points.total': 'total_points_calc',
    'stats.raids.Empty': 'empty_raids',
    'stats.raids.successful': 'successful_raids',
    'stats.raids.super_raids': 'super_raids',
    'stats.raids.total': 'total_raids',
    'stats.raids.unsuccessful': 'unsuccessful_raids',
    'stats.tackles.successful': 'successful_tackles',
    'stats.tackles.super_tackles': 'super_tackles',
    'stats.tackles.total': 'total_tackles',
    'stats.tackles.unsuccessful': 'unsuccessful_tackles'
}
df_teams = df_teams.rename(columns=col_rename)

# Impute NaN super_raids and super_tackles with 0
df_teams['super_raids'] = df_teams['super_raids'].fillna(0).astype('int8')
df_teams['super_tackles'] = df_teams['super_tackles'].fillna(0).astype('int8')

# Standardize franchise names
df_teams['team_name'] = df_teams['team_name'].replace(team_name_corrections).str.strip()

# Downcast integer types
int_cols = ['team_id', 'match_id', 'score', 'all_outs_inflicted', 'all_out_points', 'extra_points',
            'raid_bonus_points', 'raid_points_total', 'raid_touch_points', 'tackle_points_total',
            'empty_raids', 'successful_raids', 'total_raids', 'unsuccessful_raids',
            'successful_tackles', 'total_tackles', 'unsuccessful_tackles']
for col in int_cols:
    if col in df_teams.columns:
        df_teams[col] = df_teams[col].fillna(0).astype('int16')

print(f"Cleaned Teams Shape: {df_teams.shape}")
print(f"Teams duplicates in (match_id, team_id): {df_teams.duplicated(subset=['match_id', 'team_id']).sum()}")
df_teams[['match_id', 'team_id', 'team_name', 'score', 'raid_points_total', 'tackle_points_total', 'super_raids', 'super_tackles']].head(5)


Cleaned Teams Shape: (1368, 26)
Teams duplicates in (match_id, team_id): 0


,match_id,team_id,team_name,score,raid_points_total,tackle_points_total,super_raids,super_tackles
0,1,3,Jaipur Pink Panthers,28,25,1,0,0
1,1,5,U Mumba,44,28,12,0,0
2,10,2,Dabang Delhi K.C.,39,26,9,0,0
3,10,4,Bengal Warriors,40,25,11,0,0
4,100,2,Dabang Delhi K.C.,45,23,15,0,0


## 4. Clean Players Dataset & Role Imputation
Raw player records lack role categorization (`player_role` is entirely null). We combine official positions from historical season tournament records (`player_roles.csv`) with career raid vs. tackle volume logic to ensure complete, accurate role assignments.

In [5]:
df_players = df_players_raw.copy()

# Drop legacy index columns and null role columns
drop_cols = [c for c in ['Unnamed: 0', 'index', 'player_short_name', 'player_skill', 'player_role'] if c in df_players.columns]
df_players = df_players.drop(columns=drop_cols)

# Standardize player name
df_players['player_name'] = df_players['player_name'].str.strip()

# Build official role lookup from player_roles.csv
official_role_map = dict(zip(df_roles_raw['player_id'], df_roles_raw['position_name']))
df_players['official_position'] = df_players['player_id'].map(official_role_map)

# Inferred role logic based on career activity
career_stats = df_players.groupby('player_id')[['player_raids_total', 'player_tackles_total']].sum()

def infer_player_role(row):
    r, t = row['player_raids_total'], row['player_tackles_total']
    if r >= 10 and t >= 10:
        return 'All Rounder'
    elif r > t:
        return 'Raider'
    else:
        return 'Defender'

inferred_role_map = career_stats.apply(infer_player_role, axis=1).to_dict()
df_players['detailed_position'] = df_players['official_position'].fillna(df_players['player_id'].map(inferred_role_map))

# Map to 3 primary broad categories: Raider, Defender, All Rounder
def generalize_position(pos):
    pos_str = str(pos).lower()
    if 'raider' in pos_str:
        return 'Raider'
    elif 'all rounder' in pos_str or 'all-rounder' in pos_str:
        return 'All Rounder'
    else:
        return 'Defender'

df_players['player_category'] = df_players['detailed_position'].apply(generalize_position)

# Downcast integer metrics
numeric_player_cols = [c for c in df_players.columns if c.startswith('player_') and c not in ['player_name', 'player_played', 'player_captain', 'player_on_court', 'player_on_starter', 'player_category']]
for col in numeric_player_cols:
    df_players[col] = df_players[col].fillna(0).astype('int16')

df_players['player_id'] = df_players['player_id'].astype('int32')
df_players['match_id'] = df_players['match_id'].astype('int32')
df_players['team_id'] = df_players['team_id'].astype('int32')

print("Player role distribution:")
print(df_players['player_category'].value_counts())
print(f"Players duplicates on (match_id, player_id): {df_players.duplicated(subset=['match_id', 'player_id']).sum()}")
df_players[['player_id', 'player_name', 'player_category', 'detailed_position', 'player_total_points', 'player_raids_total', 'player_tackles_total']].head(5)


Player role distribution:
player_category
Defender       6972
Raider         6871
All Rounder    3049
Name: count, dtype: int64
Players duplicates on (match_id, player_id): 0


,player_id,player_name,player_category,detailed_position,player_total_points,player_raids_total,player_tackles_total
0,15,Sanjay Shrestha,Defender,Defender,0,0,0
1,96,Rohit Rana,Defender,"Defender, left cover",0,1,0
2,72,Navneet Gautam,Defender,"Defender, left corner",0,1,0
3,52,Jasvir Singh,Raider,Raider,12,12,10
4,22,Wei Yang,Raider,Raider,0,0,0


## 5. Clean Events Dataset
Events log every raid and tackle action. We remove duplicate play logs, drop empty review tracking fields, impute missing points with 0, and cast tactical flags (`do_or_die`, `super_raid`, `super_tackle`) to strictly typed booleans.

In [6]:
df_events = df_events_raw.copy()

# Drop dead review tracking columns and index
dead_review_cols = [c for c in df_events.columns if c.startswith('review.') or c in ['Unnamed: 0', 'reason', 'reason_id', 'status_id']]
df_events = df_events.drop(columns=dead_review_cols)

# Deduplicate identical event entries on (match_id, event_no)
pre_dedup = len(df_events)
df_events = df_events.drop_duplicates(subset=['match_id', 'event_no'], keep='first')
post_dedup = len(df_events)
print(f"Deduplicated event records: {pre_dedup - post_dedup} rows removed.")

# Normalize boolean columns
bool_cols = ['do_or_die', 'super_raid', 'super_tackle']
for col in bool_cols:
    df_events[col] = df_events[col].fillna(False).replace({'0': False, '1': True, 0: False, 1: True}).astype(bool)

# Impute numerical points with 0
point_cols = ['defending_all_out_points', 'defending_bonus_points', 'defending_capture_points',
              'defending_points', 'defending_technical_points', 'raid_all_out_points',
              'raid_bonus_points', 'raid_points', 'raid_technical_points', 'raid_touch_points']
for col in point_cols:
    df_events[col] = df_events[col].fillna(0).astype('int8')

# Standardize IDs
id_cols = ['match_id', 'event_id', 'event_no']
for col in id_cols:
    df_events[col] = df_events[col].astype('int32')

for opt_id in ['raider_id', 'defender_id', 'raiding_team_id', 'defending_team_id', 'player_id', 'team_id']:
    if opt_id in df_events.columns:
        df_events[opt_id] = df_events[opt_id].astype('Int32')

print(f"Cleaned Events Shape: {df_events.shape}")
print("Event type distribution:")
print(df_events['event'].value_counts())
df_events[['match_id', 'event_no', 'clock', 'event', 'raid_points', 'defending_points', 'do_or_die', 'super_raid', 'super_tackle']].head(5)


Deduplicated event records: 383 rows removed.


Cleaned Events Shape: (66185, 29)


Event type distribution:
event
Empty Raid                   24951
Successful Raid              19835
Unsuccessful Raid            13119
Substitution                  4733
TimeOut                       2588
Green Card                     657
Non Raid Technical Points      224
Yellow Card                     48
Raider and Anti Out             27
Red Card                         3
Name: count, dtype: int64


,match_id,event_no,clock,event,raid_points,defending_points,do_or_die,super_raid,super_tackle
0,1,1,0:33,Successful Raid,1,0,False,False,False
1,1,2,0:43,Empty Raid,0,0,False,False,False
2,1,3,0:48,Unsuccessful Raid,1,1,False,False,False
3,1,4,1:42,Unsuccessful Raid,0,1,False,False,False
4,1,5,2:19,Empty Raid,0,0,False,False,False


## 6. Data Validation Checks & Export
We run domain assertions and export clean datasets to `data/cleaned/`.

In [7]:
# Assertions
assert df_matches['match_id'].duplicated().sum() == 0, "Duplicate match_ids detected!"
assert df_teams.duplicated(subset=['match_id', 'team_id']).sum() == 0, "Duplicate team entries detected!"
assert df_players.duplicated(subset=['match_id', 'player_id']).sum() == 0, "Duplicate player match entries detected!"
assert df_events['raid_points'].min() >= 0, "Negative raid points found!"
assert df_events['defending_points'].min() >= 0, "Negative defending points found!"

# Export cleaned files
matches_path = os.path.join(CLEAN_DIR, 'cleaned_matches.csv')
teams_path = os.path.join(CLEAN_DIR, 'cleaned_teams.csv')
players_path = os.path.join(CLEAN_DIR, 'cleaned_players.csv')
events_path = os.path.join(CLEAN_DIR, 'cleaned_events.csv')

df_matches.to_csv(matches_path, index=False)
df_teams.to_csv(teams_path, index=False)
df_players.to_csv(players_path, index=False)
df_events.to_csv(events_path, index=False)

print("Export Summary:")
for name, path in [('Matches', matches_path), ('Teams', teams_path),
                   ('Players', players_path), ('Events', events_path)]:
    sz = os.path.getsize(path) / (1024 * 1024)
    print(f" - {name:8s}: {sz:.2f} MB saved to {path}")

print("Data cleaning & standardization complete! All constraints verified.")


Export Summary:
 - Matches : 0.15 MB saved to ../data/cleaned/cleaned_matches.csv
 - Teams   : 0.10 MB saved to ../data/cleaned/cleaned_teams.csv
 - Players : 1.89 MB saved to ../data/cleaned/cleaned_players.csv
 - Events  : 7.67 MB saved to ../data/cleaned/cleaned_events.csv
Data cleaning & standardization complete! All constraints verified.
